# Classifying Autism vs. Control from Brain ConnectivityA simplified project: no raw brain images, no atlases, no custom signalprocessing. We use ABIDE data that is ALREADY reduced to per-region timeseries, and turn the rest into a standard tabular ML problem.**Pipeline:** download precomputed ROI time series -> compute a functionalconnectivity matrix per subject -> flatten into features -> train aclassifier -> evaluate with cross-validation -> interpret top features.

## Step 0: Install dependencies (run once)If any import fails below, run this cell first (uncomment it), thenrestart the kernel and try again.

In [ ]:
# import sys# !{sys.executable} -m pip install nilearn numpy pandas scikit-learn

## Step 1: Import librariesOnly common, easy-to-install packages. `nilearn` is used ONLY to downloadthe data for us -- we won't touch its imaging-specific tools (no masker,no atlas plotting).

In [ ]:
import numpy as npimport pandas as pdfrom nilearn import datasetsfrom sklearn.model_selection import cross_val_score, StratifiedKFoldfrom sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import make_pipeline

## Step 2: Download precomputed ROI time seriesSetting `derivatives=["rois_cc200"]` tells nilearn to download data thathas ALREADY been reduced to ~200 regions (using the Craddock 200 atlas)by the people who preprocessed ABIDE. This skips the atlas/masker stepsentirely -- you get a plain 2D array (timepoints x regions) per subject,ready to use.This will download real data over the network the first time -- expectit to take a few minutes depending on your connection.

In [ ]:
abide = datasets.fetch_abide_pcp(    n_subjects=50,    pipeline="cpac",    derivatives=["rois_cc200"],)roi_time_series = abide.rois_cc200          # list of arrays: (timepoints, ~200 regions)phenotypic = pd.DataFrame(abide.phenotypic)  # subject info, incl. diagnosis# DX_GROUP: 1 = Autism, 2 = Controlprint(f"Number of subjects downloaded: {len(roi_time_series)}")print(f"Shape of first subject's data: {roi_time_series[0].shape}")print(phenotypic["DX_GROUP"].value_counts())

## Step 3: Turn each subject's time series into a connectivity matrixFor each subject, we compute the correlation between every pair ofregions. If region A and region B tend to go up and down together overtime, they're "functionally connected." `np.corrcoef` does this for anentire matrix in one line -- no custom math needed.The result per subject is a (n_regions x n_regions) correlation matrix.It's symmetric (corr of A-B equals corr of B-A), so we only need theupper triangle -- that becomes our list of features for that subject.

In [ ]:
def connectivity_features(ts):    """    Convert a (timepoints x regions) time series into a flattened    upper-triangle correlation vector -- this becomes one subject's    feature row.    """    corr_matrix = np.corrcoef(ts.T)  # transpose: rows=regions, cols=timepoints    # Get indices of the upper triangle, excluding the diagonal (which is    # always 1.0 -- a region is always perfectly correlated with itself)    upper_indices = np.triu_indices_from(corr_matrix, k=1)    return corr_matrix[upper_indices]

In [ ]:
feature_rows = []valid_indices = []for i, ts in enumerate(roi_time_series):    try:        features = connectivity_features(ts)        feature_rows.append(features)        valid_indices.append(i)    except Exception as e:        print(f"Subject {i} skipped: {e}")X = np.array(feature_rows)                       # shape: (n_subjects, n_connections)y = phenotypic.iloc[valid_indices]["DX_GROUP"].values  # 1 = ASD, 2 = Controlprint(f"Feature matrix shape: {X.shape}")print(f"Labels shape: {y.shape}")

## Step 4: Train and evaluate a classifierWe use logistic regression -- a simple, interpretable model, and areasonable baseline for this kind of high-dimensional-features/small-sample-size problem (you'll likely have ~50 subjects but ~20,000connectivity features, so simple + regularized models generalizebetter than complex ones here).`StandardScaler` z-scores each feature (mean 0, std 1) before fitting --logistic regression is sensitive to feature scale, so this matters.We evaluate with 5-fold cross-validation instead of a single train/testsplit, since with only ~50 subjects, one split could easily bemisleadingly lucky or unlucky. Cross-validation trains and tests 5separate times on different slices of the data and averages the result,giving a much more honest estimate of real-world accuracy.

In [ ]:
model = make_pipeline(    StandardScaler(),    LogisticRegression(max_iter=1000, C=0.1)  # C=0.1 = fairly strong regularization)cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")print(f"Cross-validated accuracy per fold: {scores}")print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")# Compare to a "dumb" baseline: always predicting the majority classmajority_class_fraction = max(np.mean(y == 1), np.mean(y == 2))print(f"Baseline (always guess majority class): {majority_class_fraction:.3f}")

## Step 5: Interpret which connections matter mostFit the model on all the data (not cross-validated this time) and lookat which connectivity features got the largest weights. Large positiveor negative weights mean that connection strongly influenced themodel's prediction in one direction or the other.

In [ ]:
model.fit(X, y)logreg = model.named_steps["logisticregression"]coefficients = logreg.coef_[0]top_indices = np.argsort(np.abs(coefficients))[::-1][:10]print("Top 10 most influential region-pair connections:")for idx in top_indices:    print(f"  connection #{idx}: weight = {coefficients[idx]:.4f}")

## Step 6: Write up honestlyThings to report in your README:- Your cross-validated accuracy vs. the majority-class baseline -- if  they're close, the model isn't learning much real signal, and that's  an honest, useful thing to report (small samples + high dimensional  features is a genuinely hard regime, and published papers wrestle  with this too)- How many subjects and connections you used- That "top connections" are exploratory, not confirmed findings -- with  ~20,000 features and ~50 subjects, some will look important by chance.  A more rigorous next step (mention this as future work) would be  permutation testing to check whether the top weights are statistically  meaningful or not.